# Student Marks Predictor

Welcome to the improved student marks predictor notebook.

This notebook trains a simple linear regression model on student study data, shows model performance, and provides an interactive UI to estimate marks based on study hours and course load.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display, Markdown
import ipywidgets as widgets

sns.set(style="whitegrid")

## 1. Load data and inspect

We load the dataset and review the first examples, plus a quick summary of the features.

In [ ]:
# Load dataset
file_path = "Student_Marks.csv"
data = pd.read_csv(file_path)

# Basic dataset overview
print(f"Dataset loaded: {data.shape[0]} rows and {data.shape[1]} columns")
display(data.head())

display(data.describe())

## 2. Data visualization

Visualize how study time and number of courses relate to marks.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

sns.scatterplot(x="time_study", y="Marks", data=data, ax=axes[0], color="#2a9d8f")
axes[0].set_title("Marks vs Study Time")
axes[0].set_xlabel("Study Hours")
axes[0].set_ylabel("Marks")

sns.scatterplot(x="number_courses", y="Marks", data=data, ax=axes[1], color="#e76f51")
axes[1].set_title("Marks vs Number of Courses")
axes[1].set_xlabel("Courses")
axes[1].set_ylabel("Marks")

plt.tight_layout()
plt.show()

## 3. Train the regression model

We use `number_courses` and `time_study` as predictors for `Marks`.

In [ ]:
# Prepare features and target
X = data[["number_courses", "time_study"]]
y = data["Marks"]

# Train model
model = LinearRegression()
model.fit(X, y)

# Predictions and metrics
predictions = model.predict(X)
mae = mean_absolute_error(y, predictions)
mse = mean_squared_error(y, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y, predictions)

# Display model performance
model_metrics = {
    "Mean Absolute Error": mae,
    "Mean Squared Error": mse,
    "Root MSE": rmse,
    "R² Score": r2,
}

metric_text = "\n".join([f"- **{name}:** {value:.2f}" for name, value in model_metrics.items()])

display(Markdown(f"### Model performance\n{metric_text}"))

display(Markdown(f"### Regression equation\nMarks = {model.intercept_:.2f} + {model.coef_[0]:.2f}×Courses + {model.coef_[1]:.2f}×Study hours"))

# Actual vs predicted plot
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y, y=predictions, color="#264653")
plt.plot([y.min(), y.max()], [y.min(), y.max()], "--", color="gray")
plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")
plt.title("Actual vs Predicted Marks")
plt.tight_layout()
plt.show()

## 4. Interactive predictor

Use the slider controls to choose study hours and number of courses, and see the predicted marks instantly.

In [ ]:
# Create UI widgets
courses_slider = widgets.IntSlider(value=2, min=0, max=10, step=1, description="Courses", style={"description_width": "120px"})
hours_slider = widgets.FloatSlider(value=5.0, min=0.0, max=12.0, step=0.1, description="Study Hours", style={"description_width": "120px"})

output = widgets.Output()


def update_prediction(change=None):
    with output:
        output.clear_output()
        courses = courses_slider.value
        hours = hours_slider.value
        predicted_marks = model.predict([[courses, hours]])[0]
        display(Markdown(f"### Predicted Marks: **{predicted_marks:.1f}**"))
        display(Markdown(f"Input values: **{courses} courses**, **{hours:.1f} study hours**"))

courses_slider.observe(update_prediction, names="value")
hours_slider.observe(update_prediction, names="value")

ui = widgets.VBox([
    widgets.HTML("<h3>Student Marks Predictor</h3><p>Adjust the inputs below and see your predicted marks.</p>"),
    widgets.HBox([courses_slider, hours_slider]),
    output
])

update_prediction()
display(ui)

## 5. Notes

- This model uses a simple linear regression and assumes a linear relationship between study hours, course count, and marks.
- For best results, use values within the range of the dataset.
- You can improve this notebook by exploring additional features, more advanced models, or a web app interface.